# 🎯 Module 07: Dimensionality Reduction

> **Stage:** Machine Learning  
> **Level:** Intermediate  
> **Module:** 07 — Dimensionality Reduction  
> **Date:** 15 January 2026  
> **Previous:** 06 — Clustering  
> **Next:** 08 — Model Evaluation

---

High-dimensional data is **hard to visualize, slow to train on, and often redundant**. **Dimensionality reduction** compresses it while preserving structure.

In this notebook we'll cover:

- **Feature selection** vs **dimensionality reduction**
- **PCA** — Principal Component Analysis
- **Explained variance** & choosing components
- **t-SNE** — for visualization
- **UMAP** — non-linear manifold projection
- **LDA** — supervised dimensionality reduction
- The **curse of dimensionality**
- A practical decision framework

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what dimensionality reduction is and why it matters
- Distinguish **feature selection** from **feature extraction**
- Apply **PCA** and interpret **explained variance**
- Choose the number of components by variance threshold
- Use **t-SNE** for visualization
- Use **UMAP** for structure exploration
- Apply **LDA** as a **supervised** projection
- Explain the **curse of dimensionality**
- Choose the right technique for the right problem

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

np.random.seed(42)
print('✅ Setup ready.')

---
## 🧠 1. What is Dimensionality Reduction?

**Dimensionality reduction** means reducing the number of **input features**.

```text
Dataset
100 features
     ↓
Dimensionality Reduction
     ↓
10 useful dimensions
```

### Why do this?

- Reduce **computational cost**
- Remove **redundant** information
- Reduce **noise**
- Enable **visualization**
- Reduce **overfitting** in some situations
- Make high-dimensional data **easier to understand**

---
## 🔀 2. Feature Selection vs Dimensionality Reduction

These are **related but different**.

### Feature Selection

Keeps a **subset of existing features**:

```text
Age, Salary, Height, Weight, Experience
     ↓
Age, Salary, Experience
```

### Dimensionality Reduction

Creates **new representations** from existing features:

```text
Age + Salary + Height + Weight
             ↓
       Transformation
             ↓
        Component 1
        Component 2
```

> The new dimensions may **not directly correspond** to individual original features.

---
## 📐 3. PCA — Principal Component Analysis

**PCA** transforms the original features into a smaller number of **principal components** that capture as much **variance** as possible.

### Intuition

```text
Original:

Feature 2
   │    • •
   │  • •
   │ •
   └────────── Feature 1
```

If the data mainly varies along one direction, PCA can represent that using one component:

```text
Feature 2
   │
   │   ╱
   │ ╱  → Principal Component 1
   │╱
   └──────── Feature 1
```

### Procedure

```text
Original Data
     ↓
Center / scale when appropriate
     ↓
Find directions of maximum variance
     ↓
Rank principal components
     ↓
Select top components
     ↓
Transform data
```

### Principal Component

A **principal component** is a new direction formed by a linear combination of original features:

$$PC_1 = w_1 x_1 + w_2 x_2 + w_3 x_3$$

- First component → captures the **most variance**
- Second → captures the **most remaining variance** subject to being **orthogonal** to the first
- And so on...

In [ ]:
# Load the Digits dataset (64 features)
digits = load_digits()
X = digits.data
y = digits.target

print('Shape:', X.shape)
print('Classes:', np.unique(y).tolist())

# Scale
X_scaled = StandardScaler().fit_transform(X)

# PCA to 2D
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

print('\nReduced to:', X_pca_2d.shape)
print('Explained variance ratio:', pca_2d.explained_variance_ratio_.round(4))
print(f'Total explained by 2 components: {pca_2d.explained_variance_ratio_.sum():.4f}')

In [ ]:
# Visualize PCA projection of digits
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, label='digit')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('PCA projection of Digits dataset (64 → 2)')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 📊 4. Explained Variance

You can inspect how much variance each component explains:

```python
pca.explained_variance_ratio_
```

Example:

```text
PC1 → 0.60
PC2 → 0.25
PC3 → 0.10
PC4 → 0.05
```

The first two components explain:

```text
60% + 25% = 85%
```

### Choosing components by variance target

```python
pca = PCA(n_components=0.95)   # retain 95% of variance
```

In [ ]:
# Full PCA to inspect the variance curve
pca_full = PCA().fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Scree plot
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_)
axes[0].set_xlabel('Component'); axes[0].set_ylabel('Explained variance ratio')
axes[0].set_title('Scree plot')
axes[0].grid(alpha=0.3)

# Cumulative variance
axes[1].plot(range(1, len(cum_var) + 1), cum_var, marker='o', markersize=3)
axes[1].axhline(0.95, color='red', linestyle='--', label='95% variance')
axes[1].set_xlabel('Number of components'); axes[1].set_ylabel('Cumulative variance')
axes[1].set_title('Cumulative explained variance')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

n_95 = np.argmax(cum_var >= 0.95) + 1
print(f'Components needed for 95% variance: {n_95} (out of {X.shape[1]})')

In [ ]:
# PCA with automatic 95% variance target
pca_95 = PCA(n_components=0.95)
X_pca_95 = pca_95.fit_transform(X_scaled)
print(f'Reduced shape: {X_pca_95.shape}')
print(f'Retained variance: {pca_95.explained_variance_ratio_.sum():.4f}')

---
## ⚠️ 5. Why Scaling Matters for PCA

PCA is affected by **feature magnitude**.

```text
Age    → 18–80
Salary → 30,000–500,000
```

Salary has a much larger numerical scale → it can **dominate the variance calculation**.

### Common workflow

```text
Raw Features
     ↓
StandardScaler
     ↓
PCA
     ↓
Reduced Features
```

> Whether scaling is appropriate depends on the **meaning and units** of features.

---
## 🧭 6. t-SNE

**t-SNE (t-distributed Stochastic Neighbor Embedding)** is mainly used to **visualize** high-dimensional data in 2D or 3D.

```text
100-dimensional data
        ↓
       t-SNE
        ↓
     2D plot
```

### What it preserves

**Local neighborhood relationships** — similar points stay close.

### ⚠️ Important

- Primarily a **visualization** technique
- **Not** a general-purpose preprocessing step for production
- Sensitive to **parameters** and **random initialization**

In [ ]:
# t-SNE on the digits dataset (subsample for speed)
n_sub = 800
idx = np.random.choice(len(X_scaled), n_sub, replace=False)
X_sub, y_sub = X_scaled[idx], y[idx]

tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_sub)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sub, cmap='tab10', s=12, alpha=0.8)
plt.colorbar(scatter, label='digit')
plt.title('t-SNE projection of Digits (64 → 2)')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🌍 7. UMAP

**UMAP (Uniform Manifold Approximation and Projection)** is another **non-linear** dimensionality-reduction technique.

Like t-SNE, it's used to **visualize high-dimensional data**, but it typically:

- Preserves **local structure** well
- Also attempts to preserve **broader structure**
- Is often **faster** than t-SNE on larger datasets

```python
import umap
reducer = umap.UMAP(n_components=2, random_state=42)
X_2d = reducer.fit_transform(X)
```

In [ ]:
# Try UMAP if installed (pip install umap-learn)
try:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = reducer.fit_transform(X_sub)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=y_sub, cmap='tab10', s=12, alpha=0.8)
    plt.colorbar(scatter, label='digit')
    plt.title('UMAP projection of Digits (64 → 2)')
    plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print('UMAP not installed. Install with: pip install umap-learn')

---
## 🏷️ 8. LDA — Linear Discriminant Analysis

There are **two different meanings** of "LDA" in ML:

1. **Latent Dirichlet Allocation** (topic modeling in NLP)
2. **Linear Discriminant Analysis** (supervised dimensionality reduction)

Here we mean **Linear Discriminant Analysis**.

### PCA vs LDA

```text
PCA:
X only
 ↓
Find directions of high variance

LDA:
X + y
 ↓
Find directions that separate classes
```

LDA attempts to:
- **Maximize between-class** variance
- **Minimize within-class** variance

Therefore, LDA is **supervised** — it needs **labels**.

In [ ]:
# LDA on the digits dataset (needs labels)
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_scaled, y)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_lda[:, 0], X_lda[:, 1], c=y, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, label='digit')
plt.title('LDA projection of Digits (supervised)')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Side-by-side comparison: PCA vs LDA on the same data
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, Xp, title in [(axes[0], X_pca_2d, 'PCA (unsupervised, variance)'),
                      (axes[1], X_lda,     'LDA (supervised, class separation)')]:
    sc = ax.scatter(Xp[:, 0], Xp[:, 1], c=y, cmap='tab10', s=8, alpha=0.7)
    ax.set_title(title)
    ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## ⚖️ 9. PCA vs t-SNE vs UMAP vs LDA

| Technique | Type | Uses Labels? | Main Use |
| --- | --- | :-: | --- |
| **PCA** | Linear | ❌ | Compression / preprocessing |
| **t-SNE** | Nonlinear | ❌ | Visualization |
| **UMAP** | Nonlinear | Usually ❌ | Visualization / structure exploration |
| **LDA** | Linear | ✅ | Class separation |

### Easy memory trick

```text
PCA   → Preserve variance
t-SNE → Preserve local neighborhoods
UMAP  → Preserve manifold / local structure
LDA   → Separate classes
```

---
## 🔥 10. The Curse of Dimensionality

As the number of dimensions increases, many ML problems become **harder**:

- More computation
- Data becomes **sparse**
- Greater distance between observations
- More difficult visualization
- Increased risk of **overfitting**

This is the **curse of dimensionality**.

> Dimensionality reduction can sometimes help by creating a more **compact representation**.

In [ ]:
# Demonstrate: as dimensions grow, points become farther apart
np.random.seed(42)
dims = [1, 2, 5, 10, 50, 100, 500, 1000]
avg_distances = []

for d in dims:
    A = np.random.rand(200, d)
    # distance from each point to the mean
    dists = np.linalg.norm(A - A.mean(axis=0), axis=1)
    avg_distances.append(dists.mean())

plt.figure(figsize=(7, 4))
plt.plot(dims, avg_distances, marker='o')
plt.xscale('log')
plt.xlabel('Number of dimensions (log)'); plt.ylabel('Mean distance to center')
plt.title('Curse of dimensionality — distances grow with dimension')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🧪 11. Practical Workflow

### For modeling

```text
100 Features
     ↓
Remove irrelevant features
     ↓
Scale when appropriate
     ↓
PCA
     ↓
20 Components
     ↓
ML Model
```

### For visualization

```text
100 Features
     ↓
PCA / UMAP / t-SNE
     ↓
2D
     ↓
Plot
     ↓
Inspect patterns
```

> Don't reduce dimensions just because you can — **check whether the reduced representation actually helps your task**.

---
## 📝 12. Practice Checklist

Use **Breast Cancer** or **Digits**:

- [ ] Standardize features
- [ ] Apply PCA
- [ ] Reduce to 2 components
- [ ] Check explained variance
- [ ] Try retaining 90–95% variance
- [ ] Visualize the result
- [ ] Try t-SNE
- [ ] Try UMAP
- [ ] Compare PCA vs non-linear visualizations
- [ ] Apply LDA using target labels

---
## 🏋️ 13. Hands-On Exercises

### Exercise 1 — PCA Scree
Plot the scree plot for the Breast Cancer dataset. How many components explain 90% of variance? 95%?

### Exercise 2 — Scaling Impact
Run PCA on unscaled and scaled data. How does the explained variance curve differ?

### Exercise 3 — t-SNE Perplexity
Try `perplexity ∈ {5, 20, 50, 100}`. How does the visualization change?

### Exercise 4 — UMAP n_neighbors
Try `n_neighbors ∈ {5, 15, 50, 100}`. What changes in the resulting plot?

### Exercise 5 — PCA vs LDA for Classification
Reduce Digits to 2 components with PCA, then with LDA. Train a classifier on each. Which is better? Why?

### Exercise 6 — Reconstruct from PCA
Use `pca.inverse_transform()` to reconstruct digit images from 2, 5, 20, and 40 components. Visualize the reconstruction quality.

---
## 🎤 14. Interview Questions

1. **What is dimensionality reduction?**  
   Reducing the number of features while preserving useful structure.

2. **Feature selection vs dimensionality reduction?**  
   Selection keeps existing features; reduction creates new ones.

3. **What is PCA?**  
   A linear projection that finds directions of maximum variance.

4. **What is a principal component?**  
   A linear combination of original features that captures a direction of high variance.

5. **Why can scaling be important before PCA?**  
   PCA is variance-based; large-scale features dominate unless standardized.

6. **What is explained variance?**  
   The fraction of total variance captured by each principal component.

7. **How do you choose the number of PCA components?**  
   Scree plot, cumulative variance threshold (e.g. 95%), or downstream performance.

8. **What is the curse of dimensionality?**  
   As dimensions grow, data becomes sparse and distances lose meaning, making ML harder.

9. **What is t-SNE mainly used for?**  
   Visualizing high-dimensional data in 2D/3D while preserving local neighborhoods.

10. **t-SNE vs PCA?**  
    PCA is linear, unsupervised, and used for preprocessing. t-SNE is nonlinear and mainly for visualization.

11. **What is UMAP?**  
    A nonlinear technique similar to t-SNE but often faster and preserving more global structure.

12. **What is Linear Discriminant Analysis?**  
    A **supervised** linear method that finds directions separating classes.

13. **PCA vs LDA?**  
    PCA is unsupervised (max variance); LDA is supervised (max class separation).

14. **Which dimensionality-reduction techniques are supervised?**  
    LDA (and its variants). PCA, t-SNE, and UMAP are unsupervised.

---
## 🏁 15. Key Takeaways

```text
Dimensionality Reduction
          ↓
Fewer Dimensions
          ↓
Less Complexity + Easier Visualization
```

### Main techniques

```text
PCA
→ Linear
→ Unsupervised
→ Preserves variance

t-SNE
→ Nonlinear
→ Visualization
→ Preserves local neighborhoods

UMAP
→ Nonlinear
→ Visualization / structure

LDA
→ Linear
→ Supervised
→ Separates classes
```

### 🧭 Golden Rule

> **PCA is commonly useful for compact representations, while t-SNE and UMAP are primarily visualization/exploration tools. LDA is the only one here that uses labels.**

---

### 🔗 What's Next?

**Module 08 — Model Evaluation**

We've trained many models. Now we'll learn how to evaluate them **rigorously** — confusion matrices, ROC curves, cross-validation strategies, and learning curves.